# Delayed flights with a Random Forest

In this exercise you'll bring together cross validation and ensemble methods. You'll be training a Random Forest classifier to predict delayed flights, using cross validation to choose the best values for model parameters.

You'll find good values for the following parameters:

- `featureSubsetStrategy` — the number of features to consider for splitting at each node and
- `maxDepth` — the maximum number of splits along any branch.
  
Unfortunately building this model takes too long, so we won't be running the `.fit()` method on the pipeline.

The `RandomForestClassifier` class has already been imported into the session.

## Instructions

- Create a random forest classifier object.
- Create a parameter grid builder object. Add grid points for the `featureSubsetStrategy` and `maxDepth` parameters.
- Create binary classification evaluator.
- Create a cross-validator object, specifying the estimator, parameter grid and evaluator. Choose 5-fold cross validation.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flight_manipulate_columns').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M4-EnsemblesAndPipelines/4_Ensembles/dataset/flights.csv',
                         sep=',',
						 header=True,
						 inferSchema=True,
						 nullValue='NA')


flights = flights.drop('flight')
flights = flights.dropna()

from pyspark.sql.functions import round
flights = flights.withColumn('km', round(flights.mile * 1.60934, 0))\
                .drop('mile')\
				.withColumn('label', (flights.delay > 15).cast('integer'))

flights = flights.sample(0.25, seed=13)

from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=[
    'mon', 'depart', 'duration'
	], outputCol='features')
flights = assembler.transform(flights)
flights = flights.select('mon', 'depart', 'duration', 'features', 'label')

flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=17)

from pyspark.ml.evaluation import BinaryClassificationEvaluator
evaluator = BinaryClassificationEvaluator()

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
print("Subset of data from the flights DataFrame:\n")
flights.show(5, False)

Subset of data from the flights DataFrame:

+---+------+--------+-----------------+-----+
|mon|depart|duration|features         |label|
+---+------+--------+-----------------+-----+
|9  |10.33 |195     |[9.0,10.33,195.0]|0    |
|1  |8.0   |232     |[1.0,8.0,232.0]  |0    |
|11 |7.77  |60      |[11.0,7.77,60.0] |1    |
|4  |13.25 |210     |[4.0,13.25,210.0]|0    |
|3  |17.58 |265     |[3.0,17.58,265.0]|1    |
+---+------+--------+-----------------+-----+
only showing top 5 rows



In [ ]:
# Create a random forest classifier
forest = ____()

# Create a parameter grid
params = ____() \
            .____(____, ['all', 'onethird', 'sqrt', 'log2']) \
            .____(____, [2, 5, 10]) \
            .____()

# Create a binary classification evaluator
evaluator = ____()

# Create a cross-validator
cv = ____(____, ____, ____, ____)

In [4]:
# Create a random forest classifier
forest = RandomForestClassifier()

# Create a parameter grid
params = ParamGridBuilder() \
            .addGrid(forest.featureSubsetStrategy, ['all', 'onethird', 'sqrt', 'log2']) \
            .addGrid(forest.maxDepth, [2, 5, 10]) \
            .build()

# Create a binary classification evaluator
evaluator = BinaryClassificationEvaluator()

# Create a cross-validator
cv = CrossValidator(estimator = forest\
                    , estimatorParamMaps = params, evaluator = evaluator, numFolds = 5)

Excellent! A grid search can be used to optimize all of the parameters in a model pipeline.